# EDA Silver layer

In [0]:
df = spark.table("marathos.bronze.raw_marathos")

df.display()

In [0]:
print(df.columns)

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns]
)

null_counts = null_counts.collect()[0].asDict()
[(column, nulls)for column, nulls in null_counts.items() if nulls > 0]

## Checking for duplicates


In [0]:
total = df.count()

distinct = df.distinct().count()

print(f"Total rows: {total}")
print(f"Distinct rows: {distinct}")
print(f"Duplicates: {total - distinct}")

In [0]:
from pyspark.sql.functions import count

# athlete id 4033 seems like a placerholder
df.groupBy("Athlete ID", "Event name", "Year of event") \
  .count() \
  .where(col("count") > 1) \
  .orderBy("count", ascending=False) \
  .display()

In [0]:
from pyspark.sql.functions import col

(df.groupBy("Athlete ID", "Event name", "Year of event") \
  .count() 
  .where(
      (col("count") > 1) & 
      (~col("Event name").contains("Split"))
  ) 
  .orderBy("count", ascending=False) 
  .display())

In [0]:
from pyspark.sql.functions import col

# Are the duplicate rows identical? TODO: Drop these
df.groupBy("Athlete ID", "Event name", "Year of event", 
           "Athlete performance", "Athlete average speed") \
  .count() \
  .where(
      (col("count") > 1) &
      (~col("Event name").contains("Split"))
  ) \
  .orderBy("count", ascending=False) \
  .display()

In [0]:
df.select("Athlete Country").distinct().display()

In [0]:
df.select("Event dates", "Athlete year of birth", "Athlete age category").display()

In [0]:
df.select(
    col("Athlete year of birth").cast("int")
).summary("min", "max", "mean").show()

In [0]:
df.where(col("Athlete year of birth").cast("int") < 1900) \
  .orderBy("Athlete year of birth", ascending=True) \
  .display()

In [0]:
df.where(
    col("Athlete age category").isNull() !=
    col("Athlete year of birth").isNull()
).display()

In [0]:
from pyspark.sql.functions import col, when

(df.withColumn("is_multiday",
    when(col("Event dates").contains("-"), True)
    .otherwise(False)
) 
.groupBy("is_multiday") 
.count() 
.display())

In [0]:
(df.where(col("Event dates").contains("-")) 
    .groupBy("Event distance/length")
    .count()
    .orderBy("count", ascending=False)
    .display()
    )

In [0]:
(df.groupBy("Event distance/length")
   .count()
   .orderBy("count", ascending=False)
   .display()
)

In [0]:
df.where(col("Event distance/length") == "None").count()

In [0]:
df.where(col("Event distance/length").isNull()).count()

In [0]:
df.display()

In [0]:
df.groupBy("Event dates").count().orderBy("count", ascending=False).display()

In [0]:
# Used claud(LLM) for this
from pyspark.sql.functions import col, regexp_extract

# Extract both dates and compare year/month
df.where(col("Event dates").contains("-")) \
  .withColumn("part1", regexp_extract(col("Event dates"), r"(\d{2}\.\d{2}\.\d{4})", 1)) \
  .withColumn("part2", regexp_extract(col("Event dates"), r"(\d{2}\.\d{2}\.\d{4})$", 1)) \
  .where(col("part1") != col("part2")) \
  .select("Event dates", "part1", "part2") \
  .display()

# Silver Layer - Cleaning Plan

## Step 1 - Rename columns to snake_case
- Rename all columns to snake_case
- Change `/` to `_or_`
- Do this FIRST so all subsequent steps use clean column names

## Step 2 - Convert None-strings to null
- Convert all "None"-strings to actual null across all string columns

## Step 3 - Cast types
- athlete_year_of_birth → int
- athlete_average_speed → double
- event_date -> to_date and add start_date and end_date

## Step 4 - Handle placeholder IDs
- athlete_id 4033 is a placeholder value → set to null

## Step 5 - Drop duplicates
- Drop fully identical rows using dropDuplicates()
- Time-based races (24h, 6h, 48h) with different distances per row are legitimate and will survive
- Do this AFTER step 4, otherwise placeholder nulls may cause unexpected results

## Step 6 - Clean anomalies
- Set athlete_year_of_birth to null if < 1800 or > 2005
- Known examples:
  - athlete_id 819254 born 1193, raced in 2020
  - athlete_id 331151 born 2017, raced in 2018

## Step 7 - Create new columns
- distance_type: extract race type from event_distance_or_length
  - km → "km"
  - mi → "miles"
  - h / d → "time_based"
  - Etappen variants (e.g. 250km/6Etappen) → extract distance only, classify as "km"
- performance_seconds: for km/mi races, convert "4:51:39 h" → seconds
- performance_km: for time-based races, extract distance in km
- Convert miles to km where applicable

## Step 8 - Calculate age category
- Only calculate if athlete_year_of_birth, year_of_event and athlete_gender are all not null
- Do NOT calculate if athlete_gender is "Missing"
- Do this AFTER year_of_birth anomalies are cleaned (Step 6)
- Format: gender prefix + age bracket (e.g. M35, W50)

## Step 9 - Fill nulls
- athlete_club → "Missing"
- athlete_country → "Missing"
- athlete_gender → "Missing"
- athlete_age_category → "Missing"
- athlete_year_of_birth → keep as null
- athlete_average_speed → keep as null

## Step 10 - Drop original columns
- Drop athlete_performance (replaced by performance_seconds and performance_km)
- Drop event_distance_or_length (replaced by distance_type)

## Data Quality Notes

### Athlete Average Speed - Updated
- Numeric values (e.g. 10.286) → cast to double, keep in athlete_average_speed
- Time values (e.g. 07:48:42) are found in BOTH km and time-based races
  → meaning of the value differs depending on distance_type
  → set to null in athlete_average_speed
  → NOT worth creating a separate column since semantic meaning is unclear
- Null count increased from 224 → 7900 (+7676 time-format values)

### Athlete ID
- athlete_id 4033 appears up to 82 times in the same race → placeholder value, set to null

### Year of Birth Anomalies
- athlete_id 819254: born 1193, raced in 2020
- athlete_id 331151: born 2017, raced in 2018 (50km in 6:56)

### Event Distance
- Mixed formats: km, miles, time-based (h/d), Etappen variants
- 1053 rows with "None" as string value → convert to null
- Decimal distances (e.g. 165.7km, 103.4km) → treat as km

### Duplicates
- Fully identical rows → drop
- Time-based races with different distances per row → keep (legitimate)

### Event Dates
- Mixed formats: single dates (06.01.2018) and date spans (20.-21.02.2016)
- Date spans indicate multi-day races
- Left as string or extract start date only

In [0]:
import re

def to_snake_case(name):
    name = name.strip().casefold()
    name = re.sub(r"[/]+", "_or_", name)
    name = re.sub(r"[\s]+", "_", name)
    return name

def rename_columns_to_snake_case(df):
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns)

to_snake_case("Event distance/length")

## Step 1 - Snake case

In [0]:
df = rename_columns_to_snake_case(df)
df.display()

## Step 2 - "None" in strings to actual null

Checking the amount of "None" in string types

In [0]:
from pyspark.sql.functions import col, when

string_cols = [column for column, types in df.dtypes if types == "string"]

for column in string_cols:
    count = df.where(col(column) == "None").count()
    null_count = df.where(col(column).isNull()).count()
    if count > 0 or null_count > 0:
        print(f"{column}: None count :{count} , Null count:{null_count}")

Changing them to null

In [0]:
for column in string_cols:
    df = df.withColumn(
        column,
        when(col(column) == "None", None)
        .otherwise(col(column))
    )

Make sure all "None" are converted to null

In [0]:
for column in string_cols:
    count = df.where(col(column) == "None").count()
    null_count = df.where(col(column).isNull()).count()
    if count > 0 or null_count > 0:
        print(f"{column}: None count :{count} , Null count:{null_count}")

## Step 3 - Cast types

Used LLM for this part:

In [0]:
nulls = df.where(col("athlete_average_speed").isNull()).count()
print(f"athlete_average_speed nulls: {nulls}")

In [0]:
from pyspark.sql.functions import col, when, regexp_extract

df = df.withColumn("athlete_average_speed",
    when(col("athlete_average_speed").rlike(r"^\d+\.?\d*$"),
        col("athlete_average_speed").cast("double")
    ).otherwise(None)
)

In [0]:
print(f"athlete_average_speed nulls: {df.where(col('athlete_average_speed').isNull()).count()}")

In [0]:
# Check what non-numeric values existed before the cast
df_bronze = spark.table("marathos.bronze.raw_marathos")

df_bronze.where(
    ~col("Athlete average speed").rlike(r"^\d+\.?\d*$") &
    col("Athlete average speed").isNotNull()
) \
.groupBy("Athlete average speed") \
.count() \
.orderBy("count", ascending=False) \
.display()

In [0]:
df_bronze.where(
    ~col("Athlete average speed").rlike(r"^\d+\.?\d*$") &
    col("Athlete average speed").isNotNull()
) \
.select("Athlete average speed", "Event distance/length") \
.distinct() \
.display()

In [0]:
(df_bronze.where(
    col("Athlete performance").rlike(r"^\d+:\d{2}:\d{2}") &
    ~col("Athlete average speed").rlike(r"^\d+\.?\d*$") &
    col("Athlete average speed").isNotNull()
) 
.select("Athlete performance", "Athlete average speed", "Event distance/length") \
.distinct() 
.display())